# 📘 Ordenamiento Topológico en Grafos Dirigidos Acíclicos (DAG)

---

# 🌟 Introducción

En esta lección aprenderemos sobre el **ordenamiento topológico** en grafos, una herramienta fundamental para **planificar tareas** o **resolver dependencias** en proyectos. 📜

> 📌 **Modelo mental:** Imagina que quieres cocinar un platillo. No puedes freír los ingredientes antes de haberlos cortado. Las tareas deben respetar un orden lógico basado en dependencias.

---

# 🌐 ¿Qué es un Ordenamiento Topológico?

- Es una forma de **ordenar los vértices** de un grafo dirigido acíclico (**DAG**) de manera que **si existe una arista de `u` a `v`**, entonces **`u` aparece antes que `v`** en el orden.

🔵 **Sólo se puede hacer en grafos SIN ciclos.**

> 📌 **Ejemplo:**
> - Tarea A → Tarea B → Tarea C
> - Orden válido: A, B, C

---

# 🔎 ¿Cómo reconocer que un problema es de ordenamiento topológico?

Este es el paso que más se salta el estudiante que va directo al código: **antes de pensar en el algoritmo, reconoce la forma del problema.** Busca estas señales en el enunciado:

- Palabras como **"antes de"**, **"después de"**, **"depende de"**, **"requiere"**, **"prerrequisito"**, **"debe completarse antes que"**.
- Te dan un conjunto de elementos (tareas, cursos, archivos, procesos) y una lista de **relaciones de precedencia** entre pares de ellos.
- Te piden **"un orden válido"** de ejecución — no un camino entre dos nodos puntuales, no una distancia. Piden **secuenciar todo el conjunto**.

Si ves esas señales, aplica el marco de decisión que ya conoces (visto en `teoria_grafos.ipynb`): la pregunta "¿en qué orden debo hacer las cosas dado un conjunto de dependencias?" se responde con ordenamiento topológico, no con un BFS/DFS genérico de alcanzabilidad.

## ⚠️ ¿Por qué exige un grafo dirigido Y acíclico?

- **Dirigido:** porque la dependencia tiene una dirección clara — "A antes que B" no es lo mismo que "B antes que A". Si modelaras esto como no dirigido, perderías esa información y cualquier orden parecería válido.
- **Acíclico:** porque un ciclo significa una dependencia circular, y una dependencia circular **no tiene solución**. Piensa en tres tareas: *Cocinar* depende de *Comprar*, *Comprar* depende de *Pagar*, y —por un error de modelado— *Pagar* depende de *Cocinar*. ¿Cuál haces primero? No hay respuesta: cada una exige que otra se haga antes, en un círculo sin punto de entrada. Un algoritmo de ordenamiento topológico que no verifica ciclos, en el mejor caso no produce un orden completo (le faltan nodos) y en el peor caso (si está mal implementado) entra en recursión infinita.
- Por eso, **el primer paso real** antes de ordenar topológicamente no es ordenar — es **verificar que el grafo sea efectivamente un DAG** (puedes reutilizar la misma idea de detección de ciclos que ya practicaste con DFS). Si detectas un ciclo, la respuesta correcta del programa es reportar que **no existe** un orden válido, no intentar forzar uno.

---

# 🧠 ¿Cómo se hace un Ordenamiento Topológico?

Usaremos un algoritmo basado en **DFS** (Depth-First Search).

## 🛤️ Algoritmo de pensamiento (usando DFS)

1. Crea una estructura para marcar nodos visitados.
2. Crea una lista o pila para almacenar el orden final.
3. Para cada nodo no visitado:
   - Aplica DFS:
     - Marca el nodo como visitado.
     - Para cada vecino, si no ha sido visitado, aplica DFS recursivamente.
     - Cuando termines con todos los vecinos, **agrega el nodo al inicio** del orden.


## 📌 Pensamiento clave

- Piensa en "terminar" una tarea después de haber terminado todas las que dependen de ella.
- **DFS explora profundo primero**; al regresar, agregas el nodo.

---

# ✍️ Simulación a mano antes de programar

Antes de escribir código, dibuja el grafo de dependencias y simula el algoritmo con una tabla, igual que hiciste con BFS/DFS. Toma este ejemplo:

Dependencias: `Comprar → Cocinar`, `Comprar → Poner la mesa`, `Cocinar → Servir`, `Poner la mesa → Servir`.

DFS desde `Comprar` (recorriendo vecinos en el orden en que aparecen arriba), agregando cada nodo **al inicio** del orden final justo cuando terminas de visitar todos sus vecinos (es decir, al "regresar" de la llamada recursiva):

| Paso | Nodo que se termina de procesar (todos sus vecinos ya visitados) | Orden final después de este paso |
|---|---|---|
| 1 | Servir (no tiene vecinos salientes) | [Servir] |
| 2 | Cocinar (su único vecino, Servir, ya está en el orden) | [Cocinar, Servir] |
| 3 | Poner la mesa (su único vecino, Servir, ya está visitado) | [Cocinar, Poner la mesa, Servir] |
| 4 | Comprar (ya visitó Cocinar y Poner la mesa) | [Comprar, Cocinar, Poner la mesa, Servir] |

Orden topológico válido: **Comprar, Cocinar, Poner la mesa, Servir**. Nota que "Cocinar" y "Poner la mesa" podrían intercambiarse — el ordenamiento topológico no siempre es único, solo tiene que respetar las dependencias.

> ⚠️ **Bug clásico de este algoritmo:** confundir "marcar visitado" (para no reprocesar un nodo) con "agregar al orden final". Son dos momentos distintos: marcas visitado quizás mucho antes de terminar de explorar todos los vecinos de ese nodo. El nodo se agrega al orden **solo cuando ya no le queda ningún vecino por explorar** — si lo agregas al orden en el momento en que lo visitas por primera vez (en vez de cuando terminas con él), el orden queda invertido y roto.

---

# 🧩 Aplicaciones del Ordenamiento Topológico

| Contexto | Descripción |
|:--------|:------------|
| Planificación de proyectos | Definir el orden en que se deben completar las tareas |
| Compilación de programas | Resolver qué archivos deben ser compilados primero |
| Curso de estudios | Determinar el orden de materias con prerrequisitos |
| Construcción de pipelines | Ordenar procesos que dependen unos de otros |

> 📌 **Ejemplo real:** Antes de cursar "Estructuras de Datos", debes haber aprobado "Fundamentos de Programación".


---

# 📝 Mini práctica: Ordenar tareas en proyectos simples

## Ejercicio

Dado el siguiente conjunto de dependencias:

- Preparar ingredientes → Cocinar
- Cocinar → Servir
- Poner la mesa → Servir

**Antes de programar:** ¿este grafo es un DAG? Dibújalo y verifica que no haya ciclos. Luego simula el algoritmo a mano con una tabla como la del ejemplo de arriba.

Realiza un posible ordenamiento topológico.

👉 **Espacio para pensar la solución (celda de código siguiente)**

> 📌 **Tip:** Dibuja el grafo primero para visualizarlo.


---

# 🎯 Resumen

| Concepto | Resumen |
|:---------|:--------|
| Ordenamiento Topológico | Ordenar nodos de un DAG respetando las dependencias |
| Cómo reconocerlo | Enunciado habla de dependencias/prerrequisitos y pide "un orden válido" para *todo* el conjunto |
| Algoritmo clásico | DFS + insertar nodos al finalizar la visita (no al empezarla) |
| Requisito | El grafo debe ser dirigido y acíclico — si tiene un ciclo, no existe orden válido |
| Usos principales | Planificación, compilación, prerrequisitos |

---

# 📘 Ordenamiento Topológico Aplicado: Planificación de Horarios, Profesores y Aulas

---

# 🌟 Contexto del problema

Imagina que en una universidad necesitamos programar clases teniendo en cuenta:

- Algunos cursos **dependen** de otros (prerrequisitos).
- No podemos dictar un curso antes de haber completado sus prerrequisitos.
- Queremos determinar un **orden de dictado** respetando todas las dependencias.

🔵 **Modelo mental:** Cada curso es un nodo; si un curso A debe preceder a un curso B, creamos una arista de A → B.

Así, el problema se convierte en **un ordenamiento topológico** de un **grafo dirigido acíclico (DAG)**.

---

# 🧠 Escenario

Cursos a programar:

| Curso | Dependencias |
|:-----|:-------------|
| Fundamentos de Programación | Ninguna |
| Estructuras de Datos | Fundamentos de Programación |
| Bases de Datos | Fundamentos de Programación |
| Programación Web | Fundamentos de Programación |
| Sistemas Operativos | Estructuras de Datos |
| Redes | Estructuras de Datos |
| Desarrollo de Aplicaciones | Programación Web, Bases de Datos |


🔵 **Construcción del grafo:**
- Fundamentos → Estructuras
- Fundamentos → Bases de Datos
- Fundamentos → Programación Web
- Estructuras → Sistemas Operativos
- Estructuras → Redes
- Bases de Datos → Desarrollo
- Programación Web → Desarrollo


---

# 📋 Algoritmo de Pensamiento para el Ordenamiento Topológico (usando DFS)

1. Crear un conjunto para registrar cursos ya visitados.
2. Crear una lista o pila para construir el ordenamiento.
3. Para cada curso no visitado:
   - Realizar un DFS:
     - Marcar el curso como visitado.
     - Para cada curso dependiente, aplicar DFS si no ha sido visitado.
     - Al terminar con todos sus dependientes, **agregar el curso al inicio del orden**.


> 📌 **Tip:** El primero en ser completado (sin dependencias) será el último en agregarse en la pila (por la naturaleza del DFS).


---

# 🛠️ Espacio para construir manualmente el ordenamiento

👉 **Tarea:** Antes de tocar el teclado, dibuja el grafo en papel y responde: ¿tiene ciclos? ¿Cuántos nodos no tienen ningún prerrequisito (esos son buenos candidatos para empezar)? Luego, usando la tabla de simulación que viste en la introducción (nodo que se termina de procesar → orden final), propone manualmente un orden válido para programar las 7 clases. Escribe tu resultado en la celda siguiente (como comentario o como una lista de Python, sin implementar el algoritmo todavía).

---

---

# 🛠️ Espacio para implementar Ordenamiento Topológico (DFS)

👉 **Ahora intenta implementar el algoritmo!**

```python
# Tu implementación del ordenamiento topológico aquí
```

---

# 🎯 Resumen

- El ordenamiento topológico permite **programar tareas** respetando dependencias.
- Se basa en realizar un **DFS** y agregar nodos en el orden inverso de finalización.
- Es esencial para **planificación de horarios, pipelines de procesos, compilaciones** y más.

✨ ¡Planificar nunca había sido tan algorítmico! 🚀